# Build the verified dataset on Kaggle

Same script, same gate, same 582 inputs as the Colab notebook. Kaggle is here
for one reason: a 9-12 hour session that does not drop, so the run is not
restarted five times.

**Before running anything — Settings, right-hand panel:**

| setting | value | why |
|---|---|---|
| Accelerator | **GPU T4 x2** | the script uses one; the second is idle |
| Internet | **On** | the clone and the model download both need it |
| Persistence | Files only | keeps `/kaggle/working` between sessions |

Internet off is the expensive mistake: the download fails at load time, after
the session has already started spending the weekly budget.

**Nothing needs uploading.** The 582 functions come with the clone (they are
committed as `inputs.jsonl`, 246 KB) and the weights come from the Hub.

## 1. Caches to /tmp, before any import

`/kaggle/working` is a hard 20 GB and it is the only thing that survives the
session. `/tmp` is ~60 GB and is wiped. A 3 GB model download landing on the
persistent disk is the usual way that 20 GB disappears, and by the time it
matters the download has already happened.

This must be the first cell, above every import.

In [ ]:
import os

os.environ["HF_HOME"] = "/tmp/hf_cache"
os.environ["HF_DATASETS_CACHE"] = "/tmp/hf_datasets"
os.environ["TRANSFORMERS_CACHE"] = "/tmp/hf_cache"
print({k: v for k, v in os.environ.items() if k.startswith(("HF_", "TRANSFORMERS_"))})

In [ ]:
# Kaggle preinstalls its own transformers and peft, older than this needs, and
# an already-imported version wins over the one just installed. Nothing is
# imported in this cell - not even to check a version.
!pip install -q -U transformers peft accelerate
print("installed - now RESTART THE KERNEL, then run from the next cell")

## ⚠️ Restart the kernel now

Run → Restart session. Not "run all again" — an actual restart. Then continue
from the cell below; the two cells above do not need re-running except the
environment one, which is why it is repeated.

In [ ]:
import os

os.environ["HF_HOME"] = "/tmp/hf_cache"          # again: the restart cleared it
os.environ["HF_DATASETS_CACHE"] = "/tmp/hf_datasets"
os.environ["TRANSFORMERS_CACHE"] = "/tmp/hf_cache"

import torch
import transformers

# Paths, not only versions. A path under the Kaggle system site-packages means
# the restart did not take and the old version is still winning.
for module in (torch, transformers):
    print(f"{module.__name__:>14} {module.__version__:>12}  {module.__file__}")

assert torch.cuda.is_available(), "Settings -> Accelerator -> GPU T4 x2, then restart"
print("cuda:", torch.cuda.get_device_name(0))
!g++ --version | head -1   # the gate compiles and runs every candidate

## 2. The code and the inputs

Cloned rather than uploaded, so the gate deciding which rows exist is the one
with tests behind it, and `inputs.jsonl` arrives with it.

In [ ]:
import subprocess
from getpass import getpass

REPO = "safi892/fyp_training"
BRANCH = "language"
CHECKOUT = "/kaggle/working/fyp"        # working, not /tmp: it survives the session

os.chdir("/kaggle/working")             # never stand in the directory being removed
subprocess.run(["rm", "-rf", CHECKOUT], check=True)

token = getpass("GitHub token (blank if public): ").strip()
url = f"https://{token}@github.com/{REPO}.git" if token else f"https://github.com/{REPO}.git"

done = subprocess.run(["git", "clone", "-b", BRANCH, url, CHECKOUT],
                      capture_output=True, text=True)
if done.returncode != 0:
    detail = done.stderr.replace(token, "***") if token else done.stderr
    raise SystemExit(f"clone failed:\n{detail}")

os.chdir(CHECKOUT)
print(subprocess.run(["git", "log", "--oneline", "-1"], capture_output=True, text=True).stdout)

In [ ]:
import sys

sys.path.insert(0, f"{CHECKOUT}/src")
sys.path.insert(0, f"{CHECKOUT}/scripts")
from pathlib import Path

from build_optimize_dataset import drivable_recursive, extract_candidate, judge

INPUTS = f"{CHECKOUT}/my_data_annotation/recursion_optimization/inputs.jsonl"
BASE = "Qwen/Qwen2.5-Coder-1.5B-Instruct"
ADAPTER = None                          # base model: nothing to upload
OUT = "/kaggle/working/verified.jsonl"  # survives the session and lands in the output

print(len(drivable_recursive(Path(INPUTS), 40)), "functions   (expected 582)")

# The gate, on cases whose answer is known, before any GPU time is spent. If a
# wrong rewrite is not rejected here, nothing this notebook produces is worth
# keeping.
rec = "int fact(int n){ if(n<=1) return 1; return n*fact(n-1); }"
itr = "int fact(int n){ int r=1; for(int i=2;i<=n;i++) r*=i; return r; }"
assert judge(rec, itr, 10.0) is None
assert judge(rec, itr.replace("r=1", "r=0"), 10.0) == "different output"
assert judge(rec, rec, 10.0) == "still recursive"
assert extract_candidate("```cpp\nint f(){return 1;}\n```") == "int f(){return 1;}"
print("gate and extractor: ok")

## 3. Twenty functions first

The yield decides whether the rest is worth the hours. Rows kept ÷ 20 is what
582 will give. The fine-tune measured **8%** on CPU; anything clearly above that
means the base model is the better proposer, which is itself a result — it was
the fine-tune that was trained on 83% of targets that left the recursion in.

In [ ]:
def build_cmd(limit, samples=16, temperature=0.9, batch=4):
    adapter = f"--adapter {ADAPTER}" if ADAPTER else ""
    return (
        f"cd {CHECKOUT} && PYTHONPATH=src "
        f"python scripts/build_optimize_dataset.py "
        f"--backend hf --base {BASE} {adapter} --batch {batch} "
        f"--corpus {INPUTS} --out {OUT} "
        f"--limit {limit} --samples {samples} --temperature {temperature}"
    )

print(build_cmd(20))

In [ ]:
!{build_cmd(20)}

## 4. The rest

One run, because the session holds. `--limit` skips what is already done, so if
it does stop, re-running this cell continues rather than repeats — and unlike
Colab's `/content`, `/kaggle/working` survives.

Watch the weekly GPU budget: roughly 30 hours, and this should want two to five.

In [ ]:
!{build_cmd(582)}

## 5. Read every row, then take it home

The gate is re-run here in front of you. These rows were written by a process
that could have been interrupted mid-line, and the whole claim of this dataset
is that every row was executed.

In [ ]:
import json

rows = [json.loads(line) for line in open(OUT) if line.strip()]
print(f"{len(rows)} verified rows\n")

for n, row in enumerate(rows, 1):
    problem = judge(row["code"], row["improved_code"], 10.0)
    print("=" * 78)
    print(f"[{n}/{len(rows)}]   re-check: {problem or 'PASSES - compiles, runs, identical output'}")
    print("-" * 78)
    print("RECURSIVE (the author's own code)")
    print(row["code"].rstrip())
    print("-" * 78)
    print("REWRITTEN (kept only because it passed)")
    print(row["improved_code"].rstrip())
    print()

In [ ]:
# The same rows as JSONL, for copy-paste into
# my_data_annotation/recursion_optimization/verified.jsonl on the laptop.
print(f"# {len(rows)} verified rows")
for row in rows:
    print(json.dumps(row, ensure_ascii=False))

### Getting the file

`/kaggle/working/verified.jsonl` appears in the **Output** tab on the right —
download it from there, or Save Version and take it from the version's output.
The printed JSONL above is the fallback, and it is saved inside the notebook.

Keep `/kaggle/working` tidy: the clone is a few hundred files against a ~500
file cap on the output directory, so remove it before saving a version if the
save complains.

In [ ]:
!du -sh /kaggle/working/* | sort -h | tail -5
!echo "files in working:" && find /kaggle/working -type f | wc -l